In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sbhatti/financial-sentiment-analysis")

print("Path to dataset files:", path)

import pandas as pd

data = pd.read_csv(path+"\data.csv")
data.head()

Path to dataset files: C:\Users\gael1\.cache\kagglehub\datasets\sbhatti\financial-sentiment-analysis\versions\4


,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral


In [2]:
df=data.copy()
df

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral
...,...,...
5837,RISING costs have forced packaging producer Hu...,negative
5838,Nordic Walking was first used as a summer trai...,neutral
5839,"According shipping company Viking Line , the E...",neutral
5840,"In the building and home improvement trade , s...",neutral


In [3]:
features= df.drop(['Sentiment'], axis=1)
features

,Sentence
0,The GeoSolutions technology will leverage Bene...
1,"$ESI on lows, down $1.50 to $2.50 BK a real po..."
2,"For the last quarter of 2010 , Componenta 's n..."
3,According to the Finnish-Russian Chamber of Co...
4,The Swedish buyout firm has sold its remaining...
...,...
5837,RISING costs have forced packaging producer Hu...
5838,Nordic Walking was first used as a summer trai...
5839,"According shipping company Viking Line , the E..."
5840,"In the building and home improvement trade , s..."


In [5]:
labels= df['Sentiment']
labels

0       positive
1       negative
2       positive
3        neutral
4        neutral
          ...   
5837    negative
5838     neutral
5839     neutral
5840     neutral
5841    positive
Name: Sentiment, Length: 5842, dtype: object

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)
# Juntamos X_train y y_train temporalmente
train_df = X_train.copy()
train_df["label"] = y_train.values

# Número de muestras de la clase minoritaria
n_samples = train_df["label"].value_counts().min()

# Tomamos la misma cantidad de cada clase
train_balanced = (
    train_df.groupby("label", group_keys=False)
    .sample(n=n_samples, random_state=42)
)

# Separar nuevamente X e y
X_train = train_balanced.drop(columns="label")
y_train = train_balanced["label"]

print(y_train.value_counts())

label
negative    688
neutral     688
positive    688
Name: count, dtype: int64


In [12]:
# train_model.py
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

def deploy_model(train_X, train_y):
    vectorizer = TfidfVectorizer()
    X_vec = vectorizer.fit_transform(train_X)

    model = LogisticRegression()
    model.fit(X_vec, train_y)

    with open("model.pkl", "wb") as f:
        pickle.dump(model, f)

    with open("vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)

In [20]:
model = deploy_model(X_train["Sentence"], y_train)
model